# 04 — Knowledge Graph Construction & Drug Target Discovery

**Pipeline Step 4 of 5**

Constructs a **Micro-Clinical Knowledge Graph (Micro-CKG)** from the AD spatial transcriptomics data and enriches it with external biological knowledge.

### Pipeline
1. **Leiden clustering** of spots → proxy cell-type assignments
2. **Wilcoxon DE testing** across clusters → DE-filtered edges (p_adj < 0.05, |log2FC| > 0.5)
3. **Build BioCypher graph** with Gene, CellType, and Region nodes
4. **Spatial validation** via Moran's I autocorrelation
5. **Translational discovery** — mouse→human orthologs, GO enrichment, ChEMBL drug targets

### Inputs
| File | Description |
|---|---|
| `data/processed/ad_preprocessed.h5ad` | QC-filtered, normalized AnnData from Step 01 |
| `cache/stabl_results_<hash>.pkl` | Stabl results from Step 02 |
| `config/schema_config.yaml` | BioCypher schema mapping |

### Outputs
| File | Description |
|---|---|
| `cache/micro_ckg.graphml` | Serialized Micro-CKG in GraphML format |

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.spatial_pipeline import (
    load_adata, run_stabl_cached, compute_clusters,
    annotate_clusters, assign_condition_labels,
)
from src.biocypher_adapter import build_micro_ckg, save_graph, visualize_graph
from src.spatial_analytics import (
    compute_spatial_neighbors, compute_spatial_autocorr,
    run_nhood_enrichment,
)
from src.external_knowledge import (
    map_orthologs, run_go_enrichment, get_drug_targets,
)
from src.graph_analytics import (
    detect_communities, compute_centrality, find_bridge_genes, summarise_graph,
)

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CACHE_DIR = PROJECT_ROOT / "cache"

print("Imports ready.")

## 4.1 Load Data and Stabl Results

In [ ]:
adata = load_adata(DATA_PROCESSED / "ad_preprocessed.h5ad")

stabl_result = run_stabl_cached(
    adata,
    cache_dir=CACHE_DIR,
    dataset_name="geo_ad",
    label_method="condition",
    n_bootstraps=500,
    prefilter="hvg",
    n_hvgs=2000,
)

print(f"\n{stabl_result['n_selected']} Stabl-selected features loaded.")

## 4.2 Compute Leiden Clusters

Before building the graph, we need cell-type assignments. We apply Leiden community detection (a graph-based clustering algorithm) to the spot-level expression profiles. The procedure is: select highly variable genes, compute PCA (40 components), build a k-nearest-neighbor graph (k=10), and partition the graph using the Leiden algorithm at resolution 0.8.

Each resulting cluster represents a group of spots with similar expression profiles. These clusters serve as proxy cell-type labels (e.g., neuronal subtypes, glial populations) and are used to create CellType nodes in the knowledge graph. The cluster-to-region mapping assigns anatomical labels (Cortex, Hippocampus, Thalamus, etc.) based on cluster rank order, providing spatial context for each cell-type node.

In [ ]:
adata = compute_clusters(adata, n_hvgs=2000)
print(f"\nLeiden clusters: {adata.obs['leiden'].nunique()}")
print(adata.obs["leiden"].value_counts().sort_index())

# Annotate clusters with brain-region marker gene signatures
cluster_annotation = annotate_clusters(adata)
print("\nCluster annotations:")
for cid, region in sorted(cluster_annotation.items(), key=lambda x: int(x[0])):
    print(f"  Cluster {cid} → {region}")

# Assign condition labels based on ground-truth metadata
condition_labels = assign_condition_labels(adata)
n_ad = int(condition_labels.sum())
n_wt = len(condition_labels) - n_ad
print(f"\nCondition labels: {n_ad} AD / {n_wt} WT spots")

## 4.3 Build Micro-CKG (DE-Filtered, Gene-Expanded)

The knowledge graph uses **Wilcoxon rank-sum differential expression testing** to create statistically significant edges instead of simple expression thresholds.

**Gene expansion:** Stabl's strict FDP+ threshold often selects very few genes (e.g., 2). To ensure the graph is informative for drug discovery, `build_micro_ckg` automatically expands the gene list to at least `min_genes` (default 20) by including the next-highest-ranked genes by stability score. The original Stabl-selected genes are marked `is_selected=True`; expanded genes are marked `is_selected=False`.

**Filtering criteria:**
- Gene → CellType edges require adjusted p-value < 0.05 AND |log2FC| > 0.5
- Gene → Region edges are aggregated from DE-significant cluster-level associations

**Edge attributes include:**
- `log2fc` — log2 fold change from DE test
- `pval_adj` — Benjamini-Hochberg adjusted p-value
- `stability_score` — Stabl bootstrap stability score
- `mean_expression` — mean expression in the cluster

In [ ]:
schema_path = PROJECT_ROOT / "config" / "schema_config.yaml"

graph = build_micro_ckg(
    stabl_result=stabl_result,
    adata=adata,
    schema_path=schema_path,
    cluster_annotation=cluster_annotation,
    min_genes=20,
)

print(f"\nMicro-CKG:")
print(f"  Nodes: {graph.number_of_nodes()}")
print(f"  Edges: {graph.number_of_edges()}")

# Quick topology summary
summary = summarise_graph(graph)
print(f"  Density: {summary['density']:.4f}")
print(f"  Components: {summary['n_components']}")

## 4.4 Graph Analytics & Drug Discovery Dashboard

Multi-panel visualization showing:
- **Top hub genes** — the most connected biomarkers and their cell-type/region associations
- **Gene × Region heatmap** — which biomarkers are spatially specific to which brain regions
- **Centrality ranking** — PageRank identifies the most influential genes in the network
- **Edge composition** — breakdown of relationship types in the knowledge graph

In [ ]:
# Community detection & centrality
community_map = detect_communities(graph)
centrality_df = compute_centrality(graph)
bridge_df = find_bridge_genes(graph, centrality_df)

print("Top 10 hub genes by PageRank:")
gene_centrality = centrality_df[centrality_df["label"] == "gene"].head(10)
print(gene_centrality[["degree", "betweenness", "pagerank"]].to_string())

print("\nBridge genes (connect multiple graph communities):")
print(bridge_df.head(10)[["gene", "bridge_score", "n_communities_bridged", "betweenness"]].to_string(index=False))

# Multi-panel drug discovery dashboard
visualize_graph(graph, community_map=community_map, centrality_df=centrality_df)

## 4.5 Save Graph

The Micro-CKG is serialized to GraphML format, a standard XML-based graph format supported by NetworkX, Cytoscape, Neo4j, and other graph analysis tools. This file serves as the input to the LLM agent in Step 05 and can also be loaded into graph visualization software for interactive exploration.

In [ ]:
graph_path = save_graph(graph, CACHE_DIR / "micro_ckg.graphml")
print(f"\nGraph persisted: {graph_path}")
print(f"File size: {graph_path.stat().st_size / 1e3:.1f} KB")

## 4.6 Spatial Validation (Moran's I)

**Why this matters for drug development:** A biomarker is only therapeutically relevant if its spatial expression pattern is non-random. Moran's I > 0 with p < 0.05 confirms that a gene shows **spatially clustered expression** — it marks a real tissue compartment, not noise. This spatial specificity is critical for targeted drug delivery.

In [ ]:
# Build spatial neighbourhood graph
adata = compute_spatial_neighbors(adata, n_neighs=6)

# Spatial autocorrelation — test Stabl-selected genes
moran_df = compute_spatial_autocorr(
    adata,
    genes=stabl_result["selected_genes"],
    mode="moran",
    n_perms=100,
)

# Show top spatially autocorrelated genes
sig_moran = moran_df[moran_df["pval_norm"] < 0.05].sort_values("I", ascending=False)
print(f"\nTop spatially autocorrelated Stabl genes (Moran's I):")
print(f"({len(sig_moran)}/{len(stabl_result['selected_genes'])} genes spatially significant)\n")
print(sig_moran.head(15)[["I", "pval_norm"]].to_string())

## 4.7 Translational Drug Target Discovery

This is the key drug-development step. We take the spatially-validated mouse biomarker genes and:
1. **Map to human orthologs** — mouse gene symbols → human equivalents via HomoloGene
2. **Pathway enrichment** — what GO biological processes / KEGG pathways do these genes converge on?
3. **Drug target query** — which human orthologs have approved or clinical-stage drugs in ChEMBL?

This creates a direct pipeline from spatial transcriptomics → druggable targets.

In [ ]:
# Mouse → Human ortholog mapping
ortho_df = map_orthologs(stabl_result["selected_genes"])
human_genes = ortho_df["human_symbol"].dropna().tolist()
print(f"Mapped {len(human_genes)} / {len(stabl_result['selected_genes'])} genes to human orthologs")
display(ortho_df.head(10))

# GO / KEGG pathway enrichment
enrich_df = run_go_enrichment(human_genes)
if enrich_df is not None and not enrich_df.empty:
    print(f"\nTop enriched pathways ({len(enrich_df)} total):")
    display(enrich_df.head(15))
else:
    print("\nNo significant enrichment results returned.")

# ChEMBL drug target lookup
drug_df = get_drug_targets(human_genes)
if drug_df is not None and not drug_df.empty:
    print(f"\nDruggable targets found: {drug_df['gene'].nunique()} genes, {len(drug_df)} drug associations")
    display(drug_df.head(15))
else:
    print("\nNo drug targets found in ChEMBL for these genes.")